In [ ]:
import os
os.environ["TORCH_DYNAMO_DISABLE"] = "1"                          
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"                                             

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing


import jax
import jax.numpy as jnp
import jax.nn as jnn


In [ ]:
# ==========================================
                              
# ==========================================
test_tau_val = 0.01
test_c_val = 0.8
freqs = np.logspace(-2, 6, 100)                   
omega = 2 * np.pi * freqs
s = 1j * omega

                 
Y_true = 1.0 / (1.0 + (s * test_tau_val)**test_c_val)

## DL

In [ ]:


class PureJAXIPNet:
    """完全使用 jax.numpy 重写的高精度物理神经网络映射器"""
    
    def __init__(self, params_dict, n_poles=14, dtype=jnp.float32):
                                                                 
        self.params = jax.tree.map(lambda x: jnp.asarray(x, dtype=dtype), params_dict)
        self.n_poles = int(n_poles)

        # =========================================================
                                               
        # =========================================================
        @jax.jit
        def _forward_fn(params, x):
            
            def _fourier_map(x_in):
                B = params['fourier_map.B']
                x_proj = (2.0 * jnp.pi * x_in) @ B
                return jnp.concatenate([jnp.sin(x_proj), jnp.cos(x_proj)], axis=-1)

            def _layer_norm(x_in, weight, bias, eps=1e-5):
                mean = jnp.mean(x_in, axis=-1, keepdims=True)
                var = jnp.var(x_in, axis=-1, keepdims=True)
                return (x_in - mean) / jnp.sqrt(var + eps) * weight + bias

            def _res_block(x_in, prefix):
                fc1_w, fc1_b = params[f'{prefix}.fc1.weight'], params[f'{prefix}.fc1.bias']
                fc2_w, fc2_b = params[f'{prefix}.fc2.weight'], params[f'{prefix}.fc2.bias']
                ln1_w, ln1_b = params[f'{prefix}.norm1.weight'], params[f'{prefix}.norm1.bias']
                ln2_w, ln2_b = params[f'{prefix}.norm2.weight'], params[f'{prefix}.norm2.bias']

                res = x_in
                x_tmp = _layer_norm(x_in, ln1_w, ln1_b)
                x_tmp = jnn.silu(x_tmp @ fc1_w.T + fc1_b)
                
                x_tmp = _layer_norm(x_tmp, ln2_w, ln2_b)
                x_tmp = jnn.silu(x_tmp @ fc2_w.T + fc2_b)
                return x_tmp + res

            x_fourier = _fourier_map(x)
            x_concat = jnp.concatenate([x, x_fourier], axis=-1)

            in_w, in_b = params['in_layer.weight'], params['in_layer.bias']
            h = jnn.silu(x_concat @ in_w.T + in_b)

                         
            for i in range(4):
                h = _res_block(h, f'res_layers.{i}')

            out_w, out_b = params['out_layer.weight'], params['out_layer.bias']
            out = h @ out_w.T + out_b

                                                 
            log_tau_k = out[:, 0 : self.n_poles]
            log_w_k = out[:, self.n_poles :]
            
            tau_k = 10.0 ** log_tau_k
            w_k = 10.0 ** log_w_k
                                 
                                            
            w_k_sum = jnp.sum(w_k, axis=1, keepdims=True) + 1e-12        
       
            w_k = w_k / w_k_sum 
       
        
            return  tau_k, w_k

                        
        self._jit_forward = _forward_fn

    def __call__(self, x):
                                                       
        return self._jit_forward(self.params, x)




def load_jax_model(filepath, n_poles=14):
    """
    从硬盘加载权重并重建 PureJAXIPNet 模型
    """
               
    loaded_data = np.load(filepath)
    
                                             
    loaded_params = {key: loaded_data[key] for key in loaded_data.files}
    
                          
    new_model = PureJAXIPNet(loaded_params, n_poles=n_poles)
    print(f"✅ 成功从 {filepath} 加载模型并初始化 JAX 引擎！")
    
    return new_model

      
jax_model = load_jax_model('../../models/ipnet_14poles_100k_jax.npz')


In [ ]:

print(f"\n[*] 正在验证测试点 (tau={test_tau_val}, c={test_c_val})...")

                                                
x_test = torch.tensor([[np.log10(test_tau_val), test_c_val]], dtype=torch.float32)

tau_jax, w_jax = jax_model(x_test.cpu().numpy())


      
freqs = np.logspace(-2, 6, 100)
s = 1j * 2 * np.pi * freqs

    
Y_true = 1.0 / (1.0 + (s * test_tau_val)**test_c_val)

w_jax_squeezed = np.squeeze(w_jax)
tau_jax_squeezed = np.squeeze(tau_jax)


N_poles_jax = 1
w_jax_vec = w_jax_squeezed
tau_jax_vec = tau_jax_squeezed
Y_pred_jax = np.sum(
    w_jax_vec[:, None] / (1.0 + (s[None, :] * tau_jax_vec[:, None])), axis=0
)


In [ ]:

                                         
import time

num_iters = 25 * 25 * 25
x_test_np = x_test.cpu().numpy()

                                 
x_batch = np.repeat(x_test_np, num_iters, axis=0)

start_time = time.time()
tau_jax, w_jax = jax_model(x_batch)
end_time = time.time()

avg_time_per_iter = (end_time - start_time) / num_iters
print(f"JAX 模型批量拟合 {num_iters} 次，总耗时: {end_time - start_time:.4f} 秒, 平均每次: {avg_time_per_iter*1e3:.4f} ms")


## 2. 方法一：基于非负最小二乘法的 Debye 分解


In [ ]:
N_poles = 7                                

                     
def debye_response(params, s, N):
    w = params[:N]                   
    tau = params[N:]                     
    Y_pred = np.zeros_like(s, dtype=complex)
    for i in range(N):
        Y_pred += w[i] / (1.0 + s * tau[i])
    return Y_pred

                             
def objective(params, s, Y_true, N):
    Y_pred = debye_response(params, s, N)
                                                
    error = Y_pred - Y_true
    return np.concatenate([error.real, error.imag])

                          
                                           
w_init = np.ones(N_poles) / N_poles
                                        
tau_init = np.logspace(-6, 2, N_poles) 
p0 = np.concatenate([w_init, tau_init])

                                          
                       
bounds = (np.zeros(2 * N_poles), np.full(2 * N_poles, np.inf))

print(f"正在进行非线性最小二乘拟合 (N={N_poles}，同时优化 w_i 和 tau_i)...")
                                                  
res = least_squares(objective, p0, args=(s, Y_true, N_poles), bounds=bounds, method='trf', max_nfev=5000)

        
Y_pred_opt = debye_response(res.x, s, N_poles)

In [ ]:
import time

                        
N_test = 25
start = time.time()
for _ in range(N_test):
    res_tmp = least_squares(objective, p0, args=(s, Y_true, N_poles), bounds=bounds, method='trf', max_nfev=5000)
                    
end = time.time()
lsq_total_time = end - start
print(f"非线性最小二乘拟合 1000 次用时: {lsq_total_time:.4f} 秒/次")

1267


## 执行向量拟合 (Vector Fitting)

In [ ]:
import skrf as rf
from skrf.vectorFitting import VectorFitting
# ==========================================
                            
# ==========================================
print("正在执行 Vector Fitting...")
N_poles = 7                      

                                                          
freq_obj = rf.Frequency.from_f(freqs, unit='hz')
ntwk = rf.Network(frequency=freq_obj, s=Y_true.reshape(-1, 1, 1))

                                     
vf = VectorFitting(ntwk)
vf.vector_fit(n_poles_real=N_poles, n_poles_cmplx=0)

                 
Y_pred_vf = vf.get_model_response(0, 0, freqs)

In [ ]:
import time

                            
N_test = 25
start = time.time()
for _ in range(N_test):
    freq_obj = rf.Frequency.from_f(freqs, unit='hz')
    ntwk = rf.Network(frequency=freq_obj, s=Y_true.reshape(-1, 1, 1))
    vf = VectorFitting(ntwk)
    vf.vector_fit(n_poles_real=N_poles, n_poles_cmplx=0)
end = time.time()
vf_total_time = end - start
print(f"Vector Fitting 1000次拟合用时: {vf_total_time*625:.4f} 秒")



In [ ]:
                                 

from matplotlib.pyplot import tight_layout


fig, axs = plt.subplots(2, 2, figsize=(9, 4.6), dpi=120, sharex='col', 
                       gridspec_kw={"height_ratios": [2, 1]})                

lw_true, lw_ls, lw_vf, lw_jax = 2, 1.6, 1.4, 1
color_true, color_ls, color_vf, color_jax = "#222", "#e74c3c", "#27ae60", "#2980b9"


                        
ax = axs[0, 0]
ax.semilogx(freqs, Y_true.real, color=color_true, lw=lw_true, label="True")
ax.semilogx(freqs, Y_pred_opt.real, color=color_ls, ls='--', lw=lw_ls, label="LS")
ax.semilogx(freqs, Y_pred_vf.real, color=color_vf, ls='-.', lw=lw_vf, label="VF")
ax.semilogx(freqs, Y_pred_jax.real, color=color_jax, lw=lw_jax, label="IP-Net")

              
ax.text(0.02, 0.02, f"tau={test_tau_val}, c={test_c_val}", transform=ax.transAxes,
        fontsize=16, va='bottom', ha='left', color="black", alpha=0.85)

ax.legend(fontsize=11, ncol=2, loc='upper right', frameon=False)
ax.tick_params(labelsize=12)
ax.set_ylabel("Re(Y)", fontsize=14)

                             
ax = axs[0, 1]
ax.semilogx(freqs, Y_true.imag, color=color_true, lw=lw_true, label="True (Imag)")
ax.semilogx(freqs, Y_pred_opt.imag, color=color_ls, ls='--', lw=lw_ls, label="LS")
ax.semilogx(freqs, Y_pred_vf.imag, color=color_vf, ls='-.', lw=lw_vf, label="VF")
ax.semilogx(freqs, Y_pred_jax.imag, color=color_jax, lw=lw_jax, label="JAX")
ax.tick_params(labelsize=8)
ax.set_ylabel("Im(Y)", fontsize=14)

                         
ax = axs[1, 0]
err_ls_real = 100 * np.abs(Y_true.real - Y_pred_opt.real) / (np.abs(Y_true.real))
err_vf_real = 100 * np.abs(Y_true.real - Y_pred_vf.real) / (np.abs(Y_true.real))
err_jax_real = 100 * np.abs(Y_true.real - Y_pred_jax.real) / (np.abs(Y_true.real))

ax.loglog(freqs, err_ls_real, color=color_ls, lw=lw_ls, label="LS")
ax.loglog(freqs, err_vf_real, color=color_vf, lw=lw_vf, ls='-.', label="VF")
ax.loglog(freqs, err_jax_real, color=color_jax, lw=lw_jax, label="JAX")
ax.axhline(1.0, color='k', ls=':', lw=0.8, label='1%')
        
ax.tick_params(labelsize=12)
ax.set_xlabel("Frequency [Hz]", fontsize=14)
ax.set_ylabel("RelErr [%]", fontsize=14)

                         
ax = axs[1, 1]
err_ls_imag = 100 * np.abs(Y_true.imag - Y_pred_opt.imag) / (np.abs(Y_true.imag))
err_vf_imag = 100 * np.abs(Y_true.imag - Y_pred_vf.imag) / (np.abs(Y_true.imag))
err_jax_imag = 100 * np.abs(Y_true.imag - Y_pred_jax.imag) / (np.abs(Y_true.imag))

ax.loglog(freqs, err_ls_imag, color=color_ls, lw=lw_ls, label="LS")
ax.loglog(freqs, err_vf_imag, color=color_vf, lw=lw_vf, ls='-.', label="VF")
ax.loglog(freqs, err_jax_imag, color=color_jax, lw=lw_jax, label="JAX")
ax.axhline(1.0, color='k', ls=':', lw=0.8, label='1%')
        
ax.set_xlabel("Frequency [Hz]", fontsize=14)
ax.set_ylabel("RelErr [%]", fontsize=14)

for ax in axs.flat:
    ax.grid(which="both", alpha=0.2, color="#aaa", linestyle=':')

for ax_row in axs.flat:
        ax_row.tick_params(labelsize=12)

plt.tight_layout(pad=1.2)
plt.subplots_adjust(hspace=0.08, wspace=0.3)                 

fig.savefig("4.png",dpi=300)
plt.show()
